In [ ]:
!pip install -q -U "trl==0.12.2" "transformers==4.46.3" "peft==0.13.2" "accelerate==1.0.1" bitsandbytes datasets huggingface_hub



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 29.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggi

In [ ]:
from google.colab import files
uploaded = files.upload()   # select day30_domain_corpus.jsonl here


Saving day30_domain_corpus.jsonl to day30_domain_corpus.jsonl


In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)


Enter your Hugging Face token: ··········


In [ ]:
import torch, json
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

## **Use Qwen Base Model**

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-1.5B"
DATA_PATH    = "/content/day30_domain_corpus.jsonl"
OUTPUT_DIR   = "/content/qwen2.5-1.5b-base-ai-safety-lora"
HUB_REPO     = "nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora"
MAX_SEQ_LEN  = 256


In [ ]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = dataset["train"], dataset["test"]
print(train_ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

{'text': 'In September 2021, the Secretary-General of the United Nations issued a declaration that included a call to regulate AI to ensure it is "aligned with shared global values". That same month, the PRC published ethical guidelines for AI in China. According to the guidelines, researchers must ensure that AI abides by shared human values, is always under human control, and does not endanger public safety. Also in September 2021, the UK published its 10-year National AI Strategy, which says the British government "takes the long term risk of non-aligned Artificial General Intelligence, and the unforeseeable changes that it would mean for [...] the world, seriously". The strategy describes actions to assess long-term AI risks, including catastrophic risks.'}


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # match compute_dtype to avoid the
    bnb_4bit_use_double_quant=True,          # dtype-mismatch errors you hit on Day 29
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    max_steps=75,                 # <-- num_train_epochs=3 ki jagah
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,              # <-- 10 se 5
    eval_strategy="steps",
    eval_steps=5,                 # <-- 25 se 5
    save_strategy="steps",        # <-- "epoch" se "steps" (kyunki max_steps use kar rahe, epoch-based save match nahi karega)
    save_steps=25,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)
trainer.train()

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:428: UserWarning: You passed `packing=True` to the SFTTrainer/SFTConfig, and you are training your model with `max_steps` strategy. The dataset will be iterated until the `max_steps` are reached.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
5,3.032500,2.891666
10,2.964900,2.834724
15,2.873500,2.796043
20,2.786600,2.773833
25,2.724000,2.760242
30,2.677900,2.756322
35,2.654600,2.754662
40,2.699000,2.749305
45,2.673900,2.745734
50,2.568900,2.751715


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=75, training_loss=2.6837816492716473, metrics={'train_runtime': 1676.5503, 'train_samples_per_second': 0.716, 'train_steps_per_second': 0.045, 'total_flos': 2443130933870592.0, 'train_loss': 2.6837816492716473, 'epoch': 3.409090909090909})

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print(f"Adapter pushed to https://huggingface.co/{HUB_REPO}")


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpuu5t4nz3/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Adapter pushed to https://huggingface.co/nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B"
ADAPTER_REPO = "nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)

print("Loading fine-tuned model (base + LoRA adapter)...")
ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_REPO)

Loading base model...
Loading fine-tuned model (base + LoRA adapter)...


adapter_config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
domain_prompts = [
    "Deepfake technology has recently",
    "The biggest risk of generative AI misuse is",
    "Governments are responding to synthetic media by",
    "Content authenticity standards such as C2PA are designed to",
    "Researchers studying AI alignment have found that",
]

print("=" * 70)
print("STEP 3: DOMAIN-RELEVANT PREFIXES - Base vs Fine-tuned")
print("=" * 70)
for p in domain_prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    print("[BASE]      ", generate(base_model, p))
    print("[FINE-TUNED]", generate(ft_model, p))



STEP 3: DOMAIN-RELEVANT PREFIXES - Base vs Fine-tuned

--- Prompt: "Deepfake technology has recently" ---
[BASE]       Deepfake technology has recently become a popular topic in media and tech circles. The term “deepfakes” is derived from the words deep learning, which refers to AI algorithms that mimic natural human language patterns.
In this article we will explore what DeepFakes are, how they work, and why you should be wary of
[FINE-TUNED] Deepfake technology has recently emerged as a powerful tool for creating realistic audio and video content. It is used to create convincing digital avatars, voice actors, or even entire scenes from scratch.

--- Prompt: "The biggest risk of generative AI misuse is" ---
[BASE]       The biggest risk of generative AI misuse is that it can be used to spread misinformation or manipulate public opinion, which could have serious consequences. However, the potential benefits of using AI for social good far outweigh any risks.
As we continue to develop n

In [ ]:
ood_prompts = [
    "The capital of France is",
    "To solve a quadratic equation, you first need to",
    "Photosynthesis is the process by which plants",
]

print("\n" + "=" * 70)
print("STEP 4: OUT-OF-DOMAIN PROMPTS - Catastrophic Forgetting Check")
print("=" * 70)
for p in ood_prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    print("[BASE]      ", generate(base_model, p))
    print("[FINE-TUNED]", generate(ft_model, p))


STEP 4: OUT-OF-DOMAIN PROMPTS - Catastrophic Forgetting Check

--- Prompt: "The capital of France is" ---
[BASE]       The capital of France is in the ___
A. North Hemisphere, South Temperate Zone
B. North Hemisphere, Tropical Zone
C. South Hemisphere, Southern Temperate Zone
D. South Hemisphere, Subtropical Zone
Answer:
Solution: The capital city of France is Paris, which has a latitude position between
[FINE-TUNED] The capital of France is Paris, which has a population of 2.14 million in the urban area and 2.307 million (metro) as per January 2019 estimates. The metropolitan region of Greater Paris had an estimated population of about 15 million people in 201

--- Prompt: "To solve a quadratic equation, you first need to" ---
[BASE]       To solve a quadratic equation, you first need to find the roots of the equation. The root is where x = 0 in an equation like . There are two ways to do this.\nFirst method: Factorize\nSecond Method: Completing Square
[FINE-TUNED] To solve a quadrat

# **Use Qwen Instruct Model**

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_PATH    = "/content/day30_domain_corpus.jsonl"   # matches the uploaded file from Cell 2
OUTPUT_DIR   = "/content/qwen2.5-1.5b-instruct-ai-safety-lora"
HUB_REPO     = "nooruiit-864/qwen2.5-1.5b-instruct-ai-safety-domain-lora"
MAX_SEQ_LEN  = 256   # short on purpose: our passages are ~29 words avg -> keeps VRAM/time low
# core structural difference between instruction tuning and domain adaptation:
# no prompt/response split here, just raw continuation text -> full-sequence loss.

# ------------------------------------------------------------
# CELL 5: Load dataset
# ------------------------------------------------------------
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = dataset["train"], dataset["test"]
print(train_ds[0])

# ------------------------------------------------------------
# CELL 6: Load base model in 4-bit + tokenizer
# ------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # match compute_dtype to avoid the
    bnb_4bit_use_double_quant=True,          # dtype-mismatch errors you hit on Day 29
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

# ------------------------------------------------------------
# CELL 7: Attach LoRA adapter
# ------------------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ------------------------------------------------------------
# CELL 8: Training config + train
# ------------------------------------------------------------
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    max_steps=75,                 # <-- num_train_epochs=3 ki jagah
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,              # <-- 10 se 5
    eval_strategy="steps",
    eval_steps=5,                 # <-- 25 se 5
    save_strategy="steps",        # <-- "epoch" se "steps" (kyunki max_steps use kar rahe, epoch-based save match nahi karega)
    save_steps=25,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print(f"Adapter pushed to https://huggingface.co/{HUB_REPO}")


# ============================================================
# CELL 9: EVALUATION — Base vs Fine-tuned (Instruct variant)
# ============================================================
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_REPO = "nooruiit-864/qwen2.5-1.5b-instruct-ai-safety-domain-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)

print("Loading fine-tuned model (base + LoRA adapter)...")
ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_REPO)

def generate(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# ---- Domain-relevant prefixes (AI Safety / Deepfake / Misinformation) ----
domain_prompts = [
    "Deepfake technology has recently",
    "The biggest risk of generative AI misuse is",
    "Governments are responding to synthetic media by",
    "Content authenticity standards such as C2PA are designed to",
    "Researchers studying AI alignment have found that",
]

print("=" * 70)
print("STEP 3: DOMAIN-RELEVANT PREFIXES - Base vs Fine-tuned")
print("=" * 70)
for p in domain_prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    print("[BASE]      ", generate(base_model, p))
    print("[FINE-TUNED]", generate(ft_model, p))

# ---- Out-of-domain prompts (catastrophic forgetting check) ----
ood_prompts = [
    "The capital of France is",
    "To solve a quadratic equation, you first need to",
    "Photosynthesis is the process by which plants",
]

print("\n" + "=" * 70)
print("STEP 4: OUT-OF-DOMAIN PROMPTS - Catastrophic Forgetting Check")
print("=" * 70)
for p in ood_prompts:
    print(f"\n--- Prompt: \"{p}\" ---")
    print("[BASE]      ", generate(base_model, p))
    print("[FINE-TUNED]", generate(ft_model, p))

{'text': 'In September 2021, the Secretary-General of the United Nations issued a declaration that included a call to regulate AI to ensure it is "aligned with shared global values". That same month, the PRC published ethical guidelines for AI in China. According to the guidelines, researchers must ensure that AI abides by shared human values, is always under human control, and does not endanger public safety. Also in September 2021, the UK published its 10-year National AI Strategy, which says the British government "takes the long term risk of non-aligned Artificial General Intelligence, and the unforeseeable changes that it would mean for [...] the world, seriously". The strategy describes actions to assess long-term AI risks, including catastrophic risks.'}


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

No files have been modified since last commit. Skipping to prevent empty commit.


Adapter pushed to https://huggingface.co/nooruiit-864/qwen2.5-1.5b-instruct-ai-safety-domain-lora
Loading base model...
Loading fine-tuned model (base + LoRA adapter)...


adapter_config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

STEP 3: DOMAIN-RELEVANT PREFIXES - Base vs Fine-tuned

--- Prompt: "Deepfake technology has recently" ---
[BASE]       Deepfake technology has recently become a popular topic in the news, with many high-profile celebrities and politicians using it to create fake videos. One of the most common ways that this is done involves taking one or more photos from an individual’s face (the "face" being their eyes, nose, mouth, etc.) and then
[FINE-TUNED] Deepfake technology has recently become more popular and is increasingly being used in criminal cases. A fake video of a murder victim was posted online by the defendant's family, which helped to incriminate him. In 2018, a judge dismissed charges against an attacker who used deepfakes during an incident where he

--- Prompt: "The biggest risk of generative AI misuse is" ---
[BASE]       The biggest risk of generative AI misuse is that it could be used to create fake news or propaganda. This can cause confusion, spread misinformation and damage 

In [ ]:
print(model.config._name_or_path if hasattr(model, 'config') else "check manually")

Qwen/Qwen2.5-1.5B-Instruct


# **Day 32 LLM Fine-Tuning Mini-Project & Delivery-- 20 prompt evaluation**

In [ ]:
!pip install -q -U bitsandbytes

import os
os.kill(os.getpid(), 9)

In [ ]:
EVAL_PROMPTS = [
    # ===== IN-DOMAIN (12) — AI safety / deepfake / misinformation =====
    {"id": 1, "category": "in-domain", "prompt": "Deepfake detection tools typically work by"},
    {"id": 2, "category": "in-domain", "prompt": "Synthetic media has changed journalism because"},
    {"id": 3, "category": "in-domain", "prompt": "One major challenge in fighting online misinformation is"},
    {"id": 4, "category": "in-domain", "prompt": "The EU AI Act aims to"},
    {"id": 5, "category": "in-domain", "prompt": "Voice cloning technology raises concerns about"},
    {"id": 6, "category": "in-domain", "prompt": "During elections, disinformation campaigns often"},
    {"id": 7, "category": "in-domain", "prompt": "AI alignment researchers study"},
    {"id": 8, "category": "in-domain", "prompt": "Content provenance and watermarking help by"},
    {"id": 9, "category": "in-domain", "prompt": "Social media platforms respond to deepfakes by"},
    {"id": 10, "category": "in-domain", "prompt": "A key limitation of current deepfake detectors is"},
    {"id": 11, "category": "in-domain", "prompt": "Generative adversarial networks are used to create deepfakes because"},
    {"id": 12, "category": "in-domain", "prompt": "Public trust in media has been affected by AI because"},

    # ===== OUT-OF-DOMAIN (8) — general knowledge / math / biology / coding =====
    {"id": 13, "category": "out-of-domain", "prompt": "The capital of Japan is"},
    {"id": 14, "category": "out-of-domain", "prompt": "To solve a quadratic equation, you first"},
    {"id": 15, "category": "out-of-domain", "prompt": "Photosynthesis is the process by which"},
    {"id": 16, "category": "out-of-domain", "prompt": "A simple Python function to reverse a string looks like"},
    {"id": 17, "category": "out-of-domain", "prompt": "The water cycle consists of the following stages:"},
    {"id": 18, "category": "out-of-domain", "prompt": "World War II ended in the year"},
    {"id": 19, "category": "out-of-domain", "prompt": "The human heart has four chambers, which are"},
    {"id": 20, "category": "out-of-domain", "prompt": "In economics, supply and demand determine"},
]

print(f"Total prompts: {len(EVAL_PROMPTS)} ({sum(1 for p in EVAL_PROMPTS if p['category']=='in-domain')} in-domain, {sum(1 for p in EVAL_PROMPTS if p['category']=='out-of-domain')} out-of-domain)")

Total prompts: 20 (12 in-domain, 8 out-of-domain)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
base_model.generation_config.pad_token_id = tokenizer.eos_token_id

def generate(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return full_text[len(prompt):].strip()

results = []
for item in EVAL_PROMPTS:
    base_response = generate(base_model, item["prompt"])
    results.append({
        "id": item["id"], "category": item["category"], "prompt": item["prompt"],
        "base_response": base_response,
    })
    print(f"[{item['id']}] Base done.")

del base_model
import gc; gc.collect(); torch.cuda.empty_cache()
print("Base model generation complete.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[1] Base done.
[2] Base done.
[3] Base done.
[4] Base done.
[5] Base done.
[6] Base done.
[7] Base done.
[8] Base done.
[9] Base done.
[10] Base done.
[11] Base done.
[12] Base done.
[13] Base done.
[14] Base done.
[15] Base done.
[16] Base done.
[17] Base done.
[18] Base done.
[19] Base done.
[20] Base done.
Base model generation complete.


In [ ]:
from transformers import BitsAndBytesConfig
from huggingface_hub import snapshot_download

MERGED_MODEL_DIR = snapshot_download(
    repo_id="nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora",
    allow_patterns=["*.json", "*.safetensors", "*.txt", "merges.txt", "vocab.json"],
)
print("Downloaded to:", MERGED_MODEL_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
ft_model = AutoModelForCausalLM.from_pretrained(MERGED_MODEL_DIR, quantization_config=bnb_config, device_map="auto")
ft_model.generation_config.pad_token_id = tokenizer.eos_token_id

for i, item in enumerate(EVAL_PROMPTS):
    ft_response = generate(ft_model, item["prompt"])
    results[i]["finetuned_response"] = ft_response
    print(f"[{item['id']}] Fine-tuned done.")

del ft_model
gc.collect(); torch.cuda.empty_cache()
print("Fine-tuned model generation complete.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Downloaded to: /root/.cache/huggingface/hub/models--nooruiit-864--qwen2.5-1.5b-base-ai-safety-domain-lora/snapshots/4d25b588f6f21b126ea47923601c8c0fcdefd521


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[1] Fine-tuned done.
[2] Fine-tuned done.
[3] Fine-tuned done.
[4] Fine-tuned done.
[5] Fine-tuned done.
[6] Fine-tuned done.
[7] Fine-tuned done.
[8] Fine-tuned done.
[9] Fine-tuned done.
[10] Fine-tuned done.
[11] Fine-tuned done.
[12] Fine-tuned done.
[13] Fine-tuned done.
[14] Fine-tuned done.
[15] Fine-tuned done.
[16] Fine-tuned done.
[17] Fine-tuned done.
[18] Fine-tuned done.
[19] Fine-tuned done.
[20] Fine-tuned done.
Fine-tuned model generation complete.


In [ ]:
df = pd.DataFrame(results)
# Scoring columns add karo — manually fill karogi 1-5 scale pe
for col in ["base_relevance", "base_coherence", "base_factual", "base_safety",
            "ft_relevance", "ft_coherence", "ft_factual", "ft_safety"]:
    df[col] = ""

df.to_csv("/content/day32_eval_scoring.csv", index=False)
df.head(20)


,id,category,prompt,base_response,finetuned_response,base_relevance,base_coherence,base_factual,base_safety,ft_relevance,ft_coherence,ft_factual,ft_safety
0,1,in-domain,Deepfake detection tools typically work by,analyzing the content of the video and compari...,analyzing patterns in the audio output of a sy...,,,,,,,,
1,2,in-domain,Synthetic media has changed journalism because,it is a new way of producing and consuming new...,it is easier to create and share fake news.,,,,,,,,
2,3,in-domain,One major challenge in fighting online misinfo...,that it is difficult to distinguish between fa...,the difficulty of identifying the source of th...,,,,,,,,
3,4,in-domain,The EU AI Act aims to,protect citizens and consumers from the risks ...,promote the safe and trustworthy development o...,,,,,,,,
4,5,in-domain,Voice cloning technology raises concerns about,privacy and security. Which of the following s...,"the reliability and safety of generated media,...",,,,,,,,
5,6,in-domain,"During elections, disinformation campaigns often","target specific groups, such as women, minorit...","target specific demographics, such as age, gen...",,,,,,,,
6,7,in-domain,AI alignment researchers study,"the ethical implications of AI systems, and th...",the alignment problem from a technical perspec...,,,,,,,,
7,8,in-domain,Content provenance and watermarking help by,design\n\nThe need for provenance and watermar...,identifying the source of the data and verifyi...,,,,,,,,
8,9,in-domain,Social media platforms respond to deepfakes by,implementing various measures to combat the sp...,"removing them from search results, blocking th...",,,,,,,,
9,10,in-domain,A key limitation of current deepfake detectors is,that they are trained on a small number of lab...,"that they are trained on a single speaker, and...",,,,,,,,
